In [5]:
import os
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@11'


# Verify Java is found
import subprocess
result = subprocess.run(['java', '-version'],
                       capture_output=True, text=True)
print(result.stderr)    # java -version prints to stderr
import sys
sys.path.insert(0, '../..')

import os
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType,
    StringType, TimestampType
)
from pyspark.sql.window import Window

Path('../../data/delta_lake').mkdir(parents=True, exist_ok=True)

print("✅ Imports ready")

openjdk version "11.0.31" 2026-04-21
OpenJDK Runtime Environment Homebrew (build 11.0.31+0)
OpenJDK 64-Bit Server VM Homebrew (build 11.0.31+0, mixed mode)

✅ Imports ready


In [6]:
# Create Spark Session
spark = (
    SparkSession.builder
    .appName("ProductionRecSys-ETL")
    .master("local[*]")               # use all CPU cores
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.1.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print(f"✅ Spark session created")
print(f"   Version : {spark.version}")
print(f"   App name: {spark.sparkContext.appName}")
print(f"   Cores   : {spark.sparkContext.defaultParallelism}")


✅ Spark session created
   Version : 3.5.1
   App name: ProductionRecSys-ETL
   Cores   : 10


In [7]:
# Quick sanity check
print(f"Spark version    : {spark.version}")
print(f"Cores available  : {spark.sparkContext.defaultParallelism}")
print(f"Driver memory    : {spark.sparkContext.getConf().get('spark.driver.memory')}")

# Check data file exists
import os
ratings_path = '../../data/raw/ratings.csv'
size_gb = os.path.getsize(ratings_path) / 1024**3
print(f"\nratings.csv size : {size_gb:.2f} GB")
print(f"Path exists      : {os.path.exists(ratings_path)} ✅")

Spark version    : 3.5.1
Cores available  : 10
Driver memory    : 4g

ratings.csv size : 0.66 GB
Path exists      : True ✅


In [10]:
## Define Schemas Explicitly
## Defining schemas explicitly instead of inferring them is faster and prevents type errors. 
#Production Spark pipelines always define schemas.

ratings_schema = StructType([
    StructField("userId",    IntegerType(), True),
    StructField("movieId",   IntegerType(), True),
    StructField("rating",    FloatType(),   True),
    StructField("timestamp", IntegerType(), True),
])

movies_schema = StructType([
    StructField("movieId",   IntegerType(), True),
    StructField("tmdb_id",   IntegerType(), True),
    StructField("title",     StringType(),  True),
    StructField("overview",  StringType(),  True),
    StructField("year",      IntegerType(), True),
    StructField("era",       StringType(),  True),
    StructField("vote_average", FloatType(), True),
    StructField("vote_count",   FloatType(), True),
    StructField("popularity",   FloatType(), True),
])

print("✅ Schemas defined")

✅ Schemas defined


In [11]:
# Load Full ratings.csv With Spark
RAW = '../../data/raw/'

print("Loading ratings.csv (26M rows) with PySpark...")
start = time.time()

ratings_spark = (
    spark.read
    .option("header", "true")
    .schema(ratings_schema)
    .csv(RAW + "ratings.csv")
)

# Cache in memory for repeated operations
ratings_spark.cache()
count = ratings_spark.count()

elapsed = time.time() - start
print(f"✅ Loaded {count:,} ratings in {elapsed:.1f}s")
print(f"\nSchema:")
ratings_spark.printSchema()
print(f"\nSample rows:")
ratings_spark.show(5)


Loading ratings.csv (26M rows) with PySpark...


✅ Loaded 26,024,289 ratings in 8.1s

Schema:
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- timestamp: integer (nullable = true)


Sample rows:
+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    110|   1.0|1425941529|
|     1|    147|   4.5|1425942435|
|     1|    858|   5.0|1425941523|
|     1|   1221|   5.0|1425941546|
|     1|   1246|   5.0|1425941556|
+------+-------+------+----------+
only showing top 5 rows



In [12]:
# Spark SQL Analysis

# Register as temp view for SQL queries
ratings_spark.createOrReplaceTempView("ratings")

print("Running Spark SQL queries on 26M ratings...\n")

# Query 1 — basic stats
print("── Basic Stats ──────────────────────────")
spark.sql("""
    SELECT
        COUNT(*)                    AS total_ratings,
        COUNT(DISTINCT userId)      AS unique_users,
        COUNT(DISTINCT movieId)     AS unique_movies,
        ROUND(AVG(rating), 3)       AS avg_rating,
        MIN(rating)                 AS min_rating,
        MAX(rating)                 AS max_rating
    FROM ratings
""").show()

# Query 2 — rating distribution
print("── Rating Distribution ──────────────────")
spark.sql("""
    SELECT rating,
           COUNT(*) as count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*))
                 OVER(), 2) as pct
    FROM ratings
    GROUP BY rating
    ORDER BY rating
""").show()

# Query 3 — top 10 most rated movies
print("── Top 10 Most Rated Movies ─────────────")
spark.sql("""
    SELECT movieId,
           COUNT(*)             AS rating_count,
           ROUND(AVG(rating),2) AS avg_rating
    FROM ratings
    GROUP BY movieId
    ORDER BY rating_count DESC
    LIMIT 10
""").show()

# Query 4 — user activity distribution
print("── User Activity Buckets ────────────────")
spark.sql("""
    SELECT
        CASE
            WHEN cnt < 10   THEN 'casual (< 10)'
            WHEN cnt < 50   THEN 'regular (10-50)'
            WHEN cnt < 200  THEN 'active (50-200)'
            ELSE                 'power (200+)'
        END AS activity_bucket,
        COUNT(*) AS user_count
    FROM (
        SELECT userId, COUNT(*) AS cnt
        FROM ratings
        GROUP BY userId
    )
    GROUP BY activity_bucket
    ORDER BY user_count DESC
""").show()

Running Spark SQL queries on 26M ratings...

── Basic Stats ──────────────────────────


+-------------+------------+-------------+----------+----------+----------+
|total_ratings|unique_users|unique_movies|avg_rating|min_rating|max_rating|
+-------------+------------+-------------+----------+----------+----------+
|     26024289|      270896|        45115|     3.528|       0.5|       5.0|
+-------------+------------+-------------+----------+----------+----------+

── Rating Distribution ──────────────────
+------+-------+-----+
|rating|  count|  pct|
+------+-------+-----+
|   0.5| 404897| 1.56|
|   1.0| 843310| 3.24|
|   1.5| 403607| 1.55|
|   2.0|1762440| 6.77|
|   2.5|1255358| 4.82|
|   3.0|5256722|20.20|
|   3.5|3116213|11.97|
|   4.0|6998802|26.89|
|   4.5|2170441| 8.34|
|   5.0|3812499|14.65|
+------+-------+-----+

── Top 10 Most Rated Movies ─────────────
+-------+------------+----------+
|movieId|rating_count|avg_rating|
+-------+------------+----------+
|    356|       91921|      4.05|
|    318|       91082|      4.43|
|    296|       87901|      4.17|
|    593

In [13]:
# Spark DataFrame Transformations
print("Running Spark DataFrame transformations...\n")

# Convert Unix timestamp to proper datetime
ratings_transformed = ratings_spark.withColumn(
    "timestamp",
    F.to_timestamp(F.col("timestamp").cast("long"))
)

# Add derived time columns
ratings_transformed = (
    ratings_transformed
    .withColumn("year",         F.year("timestamp"))
    .withColumn("month",        F.month("timestamp"))
    .withColumn("day_of_week",  F.dayofweek("timestamp"))
    .withColumn("hour",         F.hour("timestamp"))
)

# Add rating category
ratings_transformed = ratings_transformed.withColumn(
    "rating_category",
    F.when(F.col("rating") <= 2.0, "negative")
     .when(F.col("rating") <= 3.0, "neutral")
     .when(F.col("rating") <= 4.0, "positive")
     .otherwise("very_positive")
)

# Window function — user's rating rank over time
user_window = Window.partitionBy("userId").orderBy("timestamp")
ratings_transformed = ratings_transformed.withColumn(
    "user_rating_sequence",
    F.row_number().over(user_window)
)

print("Transformed schema:")
ratings_transformed.printSchema()
print("\nSample transformed rows:")
ratings_transformed.show(5)

# Rating category distribution
print("\nRating category distribution:")
ratings_transformed.groupBy("rating_category") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()


Running Spark DataFrame transformations...

Transformed schema:
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- rating_category: string (nullable = false)
 |-- user_rating_sequence: integer (nullable = false)


Sample transformed rows:


+------+-------+------+-------------------+----+-----+-----------+----+---------------+--------------------+
|userId|movieId|rating|          timestamp|year|month|day_of_week|hour|rating_category|user_rating_sequence|
+------+-------+------+-------------------+----+-----+-----------+----+---------------+--------------------+
|    12|   2858|   5.0|1999-11-29 16:13:53|1999|   11|          2|  16|  very_positive|                   1|
|    12|   2997|   5.0|1999-11-29 16:13:53|1999|   11|          2|  16|  very_positive|                   2|
|    12|   3077|   4.0|1999-11-29 16:13:53|1999|   11|          2|  16|       positive|                   3|
|    12|   2840|   1.0|1999-11-29 16:14:58|1999|   11|          2|  16|       negative|                   4|
|    12|   3079|   4.0|1999-11-29 16:15:22|1999|   11|          2|  16|       positive|                   5|
+------+-------+------+-------------------+----+-----+-----------+----+---------------+--------------------+
only showing top 5 

In [15]:
# Compute Full-Scale Movie Stats

print("Computing movie statistics from 26M ratings...")
start = time.time()

movie_stats_full = ratings_spark.groupBy("movieId").agg(
    F.count("rating")             .alias("rating_count"),
    F.avg("rating")               .alias("rating_mean"),
    F.stddev("rating")            .alias("rating_std"),
    F.expr("percentile(rating,0.5)").alias("rating_median"),
    F.countDistinct("userId")     .alias("unique_users"),
    F.min("rating")               .alias("rating_min"),
    F.max("rating")               .alias("rating_max"),
)

# Add log count
movie_stats_full = movie_stats_full.withColumn(
    "log_rating_count",
    F.log1p(F.col("rating_count"))
)

# Add popularity tier using ntile window function
window_all = Window.orderBy("rating_count")
movie_stats_full = movie_stats_full.withColumn(
    "popularity_ntile",
    F.ntile(4).over(window_all)
).withColumn(
    "popularity_tier",
    F.when(F.col("popularity_ntile") == 1, "cold")
     .when(F.col("popularity_ntile") == 2, "low")
     .when(F.col("popularity_ntile") == 3, "medium")
     .otherwise("hot")
)

elapsed = time.time() - start
print(f"✅ Movie stats computed in {elapsed:.1f}s")
movie_stats_full.show(10)
print(f"Total movies with ratings: "
      f"{movie_stats_full.count():,}")

Computing movie statistics from 26M ratings...
✅ Movie stats computed in 0.1s


+-------+------------+-----------+----------+-------------+------------+----------+----------+------------------+----------------+---------------+
|movieId|rating_count|rating_mean|rating_std|rating_median|unique_users|rating_min|rating_max|  log_rating_count|popularity_ntile|popularity_tier|
+-------+------------+-----------+----------+-------------+------------+----------+----------+------------------+----------------+---------------+
|  25816|           1|        3.0|      NULL|          3.0|           1|       3.0|       3.0|0.6931471805599453|               1|           cold|
|  26223|           1|        3.0|      NULL|          3.0|           1|       3.0|       3.0|0.6931471805599453|               1|           cold|
|  26254|           1|        2.5|      NULL|          2.5|           1|       2.5|       2.5|0.6931471805599453|               1|           cold|
|  26863|           1|        2.0|      NULL|          2.0|           1|       2.0|       2.0|0.6931471805599453|     

In [16]:
# Compute Full-Scale User Stats
print("Computing user statistics from 26M ratings...")
start = time.time()

user_stats_full = ratings_spark.groupBy("userId").agg(
    F.count("rating")        .alias("rating_count"),
    F.avg("rating")          .alias("rating_mean"),
    F.stddev("rating")       .alias("rating_std"),
    F.countDistinct("movieId").alias("unique_movies"),
    F.min("rating")          .alias("min_rating"),
    F.max("rating")          .alias("max_rating"),
)

# Activity tier
window_user = Window.orderBy("rating_count")
user_stats_full = user_stats_full.withColumn(
    "activity_ntile",
    F.ntile(4).over(window_user)
).withColumn(
    "activity_tier",
    F.when(F.col("activity_ntile") == 1, "inactive")
     .when(F.col("activity_ntile") == 2, "casual")
     .when(F.col("activity_ntile") == 3, "active")
     .otherwise("power")
)

elapsed = time.time() - start
print(f"✅ User stats computed in {elapsed:.1f}s")
user_stats_full.show(10)
print(f"Total users: {user_stats_full.count():,}")

Computing user statistics from 26M ratings...
✅ User stats computed in 0.1s


+------+------------+-----------+----------+-------------+----------+----------+--------------+-------------+
|userId|rating_count|rating_mean|rating_std|unique_movies|min_rating|max_rating|activity_ntile|activity_tier|
+------+------------+-----------+----------+-------------+----------+----------+--------------+-------------+
|   473|           1|        4.0|      NULL|            1|       4.0|       4.0|             1|     inactive|
| 15592|           1|        5.0|      NULL|            1|       5.0|       5.0|             1|     inactive|
| 20683|           1|        5.0|      NULL|            1|       5.0|       5.0|             1|     inactive|
| 21440|           1|        3.0|      NULL|            1|       3.0|       3.0|             1|     inactive|
| 23402|           1|        5.0|      NULL|            1|       5.0|       5.0|             1|     inactive|
| 25795|           1|        4.0|      NULL|            1|       4.0|       4.0|             1|     inactive|
| 38655|  

In [17]:
# Pandas vs PySpark benchmark

import time

PROC = '../../data/processed/'

print("PANDAS vs PYSPARK BENCHMARK")
print("=" * 50)
print("Task: compute avg rating per movie\n")

# Pandas on small dataset
print("1. Pandas on ratings_small.csv (100K rows)...")
start = time.time()
ratings_pd = pd.read_csv(PROC + 'ratings_cleaned.csv')
result_pd  = ratings_pd.groupby('movieId')['rating'].mean()
pandas_small_time = time.time() - start
print(f"   Time: {pandas_small_time:.3f}s ✅")

# Pandas on full dataset
print("\n2. Pandas on ratings.csv (26M rows)...")
start = time.time()
ratings_pd_full = pd.read_csv(
    '../../data/raw/ratings.csv',
    dtype={'userId': 'int32', 'movieId': 'int32',
           'rating': 'float32', 'timestamp': 'int32'}
)
result_pd_full = ratings_pd_full.groupby(
    'movieId')['rating'].mean()
pandas_full_time = time.time() - start
print(f"   Time: {pandas_full_time:.3f}s")

# PySpark on full dataset
print("\n3. PySpark on ratings.csv (26M rows)...")
start = time.time()
result_spark = ratings_spark.groupBy("movieId").agg(
    F.avg("rating").alias("avg_rating"))
result_spark.count()   # trigger computation
spark_time = time.time() - start
print(f"   Time: {spark_time:.3f}s ✅")

print(f"""
RESULTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Pandas  small (100K) : {pandas_small_time:.3f}s
Pandas  full  (26M)  : {pandas_full_time:.3f}s
PySpark full  (26M)  : {spark_time:.3f}s
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Speedup PySpark vs Pandas (26M):
{pandas_full_time/spark_time:.1f}x faster

Use Pandas  → small data  (< 1M rows)
Use PySpark → large data  (> 1M rows)
""")

PANDAS vs PYSPARK BENCHMARK
Task: compute avg rating per movie

1. Pandas on ratings_small.csv (100K rows)...
   Time: 0.065s ✅

2. Pandas on ratings.csv (26M rows)...
   Time: 6.786s

3. PySpark on ratings.csv (26M rows)...
   Time: 0.241s ✅

RESULTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Pandas  small (100K) : 0.065s
Pandas  full  (26M)  : 6.786s
PySpark full  (26M)  : 0.241s
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Speedup PySpark vs Pandas (26M):
28.2x faster

Use Pandas  → small data  (< 1M rows)
Use PySpark → large data  (> 1M rows)



In [ ]:
## Save to Delta Lake

DELTA = '../../data/delta_lake/'

print("Saving to Delta Lake...")

# Save ratings
(ratings_transformed
 .write
 .format("delta")
 .mode("overwrite")
 .save(DELTA + "ratings_full"))

# Save movie stats
(movie_stats_full
 .write
 .format("delta")
 .mode("overwrite")
 .save(DELTA + "movie_stats_full"))

# Save user stats
(user_stats_full
 .write
 .format("delta")
 .mode("overwrite")
 .save(DELTA + "user_stats_full"))

print("✅ Delta Lake tables saved")
print(f"""
Delta Lake tables:
  {DELTA}ratings_full/      ← 26M ratings + time features
  {DELTA}movie_stats_full/  ← per-movie stats from 26M ratings
  {DELTA}user_stats_full/   ← per-user stats from 26M ratings
""")

# Verify by reading back
print("Verification — reading back from Delta Lake:")
verify = spark.read.format("delta").load(
    DELTA + "ratings_full")
print(f"  ratings_full rows: {verify.count():,}")

Saving to Delta Lake...


[Stage 72:>                                                         (0 + 8) / 8]

In [ ]:
## Save Full Stats To Processed

